# 📝 Review Summarization with GPT
## Amazon Product Reviews — Generative AI
---
This notebook generates recommendation articles for each product meta-category
using the **OpenAI GPT API**.

### 📋 Output per category
Each generated article includes:
- 🏆 Top 3 products and key differences between them
- ⚠️ Top complaints for each of those products
- ❌ Worst product in the category and why to avoid it

### 📋 Plan
1. Load clustered reviews dataset
2. Prepare review summaries per category
3. Design prompt template
4. Generate articles with GPT
5. Save results

## 1. 📦 Libraries

In [5]:
# Core libraries
import os
import pandas as pd
import os


# OpenAI API
from openai import OpenAI
from dotenv import load_dotenv

print("✅ Libraries loaded")

✅ Libraries loaded


## 2. 🔑 API Key Setup
The API key is loaded from an environment variable — never hardcoded in the notebook.

> ⚠️ Never commit your API key to Git.  
> Add `.env` to your `.gitignore`.

In [6]:
# Load environment variables from .env file
load_dotenv("../.env")

# Load API key
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Set it in the .env file.")

client = OpenAI(api_key=api_key)
print("✅ OpenAI client initialized")

✅ OpenAI client initialized


## 3. 📥 Load Clustered Data
Loading the dataset with meta-categories assigned by the clustering model.

In [7]:
# Load clustered reviews dataset
df = pd.read_csv("../data/processed/reviews_clustered.csv")

print(f"Shape: {df.shape}")
print(f"\nMeta-categories:")
print(df["meta_category"].value_counts())
print(f"\nColumns: {df.columns.tolist()}")

Shape: (4805, 14)

Meta-categories:
meta_category
Fire Tablets               2818
Alexa Devices              1566
E-Readers & Accessories     361
E-Readers Premium            56
Streaming Devices             4
Name: count, dtype: int64

Columns: ['product_name', 'brand', 'categories', 'primary_category', 'rating', 'review_text', 'review_title', 'recommended', 'helpful_votes', 'username', 'sentiment', 'label', 'predicted_sentiment_finetuned', 'meta_category']


## 4. 🧰 Prepare Review Data per Category
For each meta-category we extract:
- Top products by average rating (min. 10 reviews)
- Sample of positive and negative reviews per product
- Overall category statistics

In [9]:
def prepare_category_data(df: pd.DataFrame, category: str) -> dict:
    """Prepare review data for a given meta-category.

    Args:
        df: Full clustered reviews DataFrame.
        category: Meta-category name.

    Returns:
        Dict with category stats, top products and sample reviews.
    """
    cat_df = df[df["meta_category"] == category].copy()

    # Product-level stats
    product_stats = (
        cat_df.groupby("product_name")
        .agg(
            num_reviews=("review_text", "count"),
            avg_rating=("rating", "mean"),
            pct_positive=("predicted_sentiment_finetuned",
                          lambda x: (x == "positive").mean())
        )
        .query("num_reviews >= 10")
        .sort_values("avg_rating", ascending=False)
        .reset_index()
    )

    # Sample reviews per product
    product_reviews = {}
    for product in product_stats["product_name"].tolist():
        prod_df = cat_df[cat_df["product_name"] == product]

        positive = (
            prod_df[prod_df["predicted_sentiment_finetuned"] == "positive"]
            ["review_text"].head(5).tolist()
        )
        negative = (
            prod_df[prod_df["predicted_sentiment_finetuned"] == "negative"]
            ["review_text"].head(5).tolist()
        )

        product_reviews[product] = {
            "positive":   positive,
            "negative":   negative,
            "avg_rating": round(
                product_stats[product_stats["product_name"] == product]
                ["avg_rating"].values[0], 2
            ),
            "num_reviews": int(
                product_stats[product_stats["product_name"] == product]
                ["num_reviews"].values[0]
            )
        }

    return {
        "category":        category,
        "total_reviews":   len(cat_df),
        "num_products":    len(product_stats),
        "product_stats":   product_stats,
        "product_reviews": product_reviews
    }


# Test with one category
test_data = prepare_category_data(df, "Fire Tablets")
print(f"Category:     {test_data['category']}")
print(f"Total reviews: {test_data['total_reviews']}")
print(f"Products (10+ reviews): {test_data['num_products']}")
print(f"\nTop products:")
print(test_data["product_stats"][
    ["product_name", "num_reviews", "avg_rating"]
].head(5).to_string())

Category:     Fire Tablets
Total reviews: 2818
Products (10+ reviews): 12

Top products:
                                                                                  product_name  num_reviews  avg_rating
0     All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 32 GB - Includes Special Offers, Magenta           40    4.675000
1  Fire HD 10 Tablet, 10.1 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Silver Aluminum           96    4.666667
2         All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Blue           45    4.622222
3     All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta          796    4.597990
4       All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 32 GB - Includes Special Offers, Black           58    4.586207


## 5. ✍️ Prompt Template
We design a structured prompt that instructs GPT to generate
a recommendation article with a specific format.

In [10]:
def build_prompt(category_data: dict) -> str:
    """Build a structured prompt for GPT article generation.

    Args:
        category_data: Dict with category stats and product reviews.

    Returns:
        Formatted prompt string.
    """
    category = category_data["category"]
    product_reviews = category_data["product_reviews"]

    # Build product summaries for the prompt
    products_text = ""
    for i, (product, reviews) in enumerate(product_reviews.items(), 1):
        avg_rating = reviews["avg_rating"]
        num_reviews = reviews["num_reviews"]
        positive_samples = " | ".join(reviews["positive"][:3])
        negative_samples = " | ".join(reviews["negative"][:3])

        products_text += f"""
Product {i}: {product}
- Average rating: {avg_rating}/5 ({num_reviews} reviews)
- Positive reviews: {positive_samples if positive_samples else 'None'}
- Negative reviews: {negative_samples if negative_samples else 'None'}
"""

    prompt = f"""
You are a tech product reviewer writing for a consumer advice website.
Based on the following Amazon customer reviews for the category "{category}",
write a recommendation article in English.

PRODUCT DATA:
{products_text}

ARTICLE REQUIREMENTS:
Write a structured article with exactly these sections:

1. CATEGORY OVERVIEW
   Brief introduction to the {category} category (2-3 sentences).

2. TOP 3 PRODUCTS
   For each of the top 3 products by rating:
   - Product name
   - Why it stands out
   - Key differences from the others
   - Best suited for (type of user)

3. TOP COMPLAINTS
   For each top 3 product, list the 2-3 most common complaints
   from negative reviews.

4. PRODUCT TO AVOID
   Name the worst product in this category and explain specifically
   why customers were disappointed, based on the negative reviews.

5. FINAL RECOMMENDATION
   One paragraph summarizing who should buy what in this category.

Keep the tone friendly, informative and helpful.
Use the actual review content to support your points.
Article length: 400-600 words.
"""
    return prompt


# Test prompt
test_prompt = build_prompt(test_data)
print(f"Prompt length: {len(test_prompt)} characters")
print(f"\nPrompt preview:")
print(test_prompt[:400])

Prompt length: 16784 characters

Prompt preview:

You are a tech product reviewer writing for a consumer advice website.
Based on the following Amazon customer reviews for the category "Fire Tablets",
write a recommendation article in English.

PRODUCT DATA:

Product 1: All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 32 GB - Includes Special Offers, Magenta
- Average rating: 4.68/5 (40 reviews)
- Positive reviews: this was a christmas present. t


## 6. 🤖 Generate Article with GPT
Testing article generation with one category before running all.
> ⏳ Each API call takes ~5-10 seconds.

In [11]:
def generate_article(client: OpenAI, prompt: str, category: str) -> str:
    """Generate a recommendation article using GPT.

    Args:
        client: OpenAI client instance.
        prompt: Structured prompt for the article.
        category: Category name for logging.

    Returns:
        Generated article as string.
    """
    print(f"Generating article for: {category}...")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert product reviewer who writes "
                    "clear, helpful, and honest consumer advice articles."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=1000,
        temperature=0.7
    )

    article = response.choices[0].message.content
    print(f"✅ Article generated ({len(article)} characters)")
    return article


# Test with Fire Tablets
test_article = generate_article(client, test_prompt, "Fire Tablets")
print("\n" + "=" * 60)
print(test_article)

Generating article for: Fire Tablets...
✅ Article generated (4349 characters)

# Fire Tablets: A Comprehensive Review

Fire Tablets offer a versatile and cost-effective way to access a variety of digital content, from e-books to streaming video. With their user-friendly interface and seamless integration with Amazon services, they are a popular choice for readers and casual tablet users alike. In this article, we will highlight some of the top-rated products in the Fire Tablets category, address common complaints, and provide recommendations for potential buyers.

## Top 3 Products

### 1. All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 32 GB - Magenta
**Why it stands out:** With an impressive average rating of 4.68 out of 5 based on 40 reviews, this tablet has garnered praise for its size, clarity, and ease of use. Users have found it to be an excellent upgrade from previous Kindle models.

**Key differences:** This model offers a larger 8-inch HD display compared to smaller Fire mode

## 7. 🚀 Generate All Articles
Generating one article per meta-category.  
> ⏳ Each API call takes ~5-10 seconds.  
> Streaming Devices is excluded — only 4 reviews, not enough data.

In [12]:
# Categories to process (excluding Streaming Devices)
categories = [
    "Fire Tablets",
    "Alexa Devices",
    "E-Readers & Accessories",
    "E-Readers Premium"
]

# Generate articles for all categories
articles = {}

for category in categories:
    print(f"\n{'=' * 60}")
    category_data = prepare_category_data(df, category)
    prompt = build_prompt(category_data)
    article = generate_article(client, prompt, category)
    articles[category] = article

print(f"\n✅ All articles generated: {len(articles)}")


Generating article for: Fire Tablets...
✅ Article generated (4149 characters)

Generating article for: Alexa Devices...
✅ Article generated (4669 characters)

Generating article for: E-Readers & Accessories...
✅ Article generated (3966 characters)

Generating article for: E-Readers Premium...
✅ Article generated (4256 characters)

✅ All articles generated: 4


## 8. 💾 Save Results
Saving articles as individual text files and as a combined CSV for the web app.

In [13]:
import os

# Create output directory
output_dir = "../data/processed/articles"
os.makedirs(output_dir, exist_ok=True)

# Save individual articles as text files
for category, article in articles.items():
    filename = category.lower().replace(" ", "_").replace("&", "and")
    filepath = f"{output_dir}/{filename}.txt"
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {category}\n\n")
        f.write(article)
    print(f"✅ Saved: {filepath}")

# Save all articles as CSV for the web app
articles_df = pd.DataFrame([
    {"category": cat, "article": art}
    for cat, art in articles.items()
])

articles_df.to_csv("../data/processed/articles_summary.csv", index=False)
print(f"\n✅ Combined CSV saved: data/processed/articles_summary.csv")
print(f"Shape: {articles_df.shape}")

✅ Saved: ../data/processed/articles/fire_tablets.txt
✅ Saved: ../data/processed/articles/alexa_devices.txt
✅ Saved: ../data/processed/articles/e-readers_and_accessories.txt
✅ Saved: ../data/processed/articles/e-readers_premium.txt

✅ Combined CSV saved: data/processed/articles_summary.csv
Shape: (4, 2)


## 9. 👀 Preview Articles
Quick preview of each generated article.

In [14]:
# Preview each generated article
for category, article in articles.items():
    print(f"\n{'=' * 60}")
    print(f"📄 {category}")
    print(f"{'=' * 60}")
    print(article[:500])
    print("...")


📄 Fire Tablets
# Fire Tablets: An In-Depth Review and Recommendation

Fire Tablets from Amazon have become popular devices for reading, browsing, and streaming, thanks to their affordability and user-friendly interface. Available in various sizes and specifications, these tablets cater to a wide range of users—from avid readers to casual browsers. In this article, we'll explore the top products in the Fire Tablets category, highlight their strengths and weaknesses, and guide you on making an informed purchase.
...

📄 Alexa Devices
# Alexa Devices: A Comprehensive Recommendation Guide

In the rapidly evolving world of smart home technology, Alexa devices have emerged as essential tools for enhancing convenience and connectivity in our daily lives. From controlling smart home devices to answering questions and playing music, these devices offer a variety of features that cater to diverse user needs. Here, we explore the top-rated Alexa devices based on customer reviews, helping you make

## 10. 📋 Summarization Summary

### Results

| Category | Reviews Used | Article Length |
|---|---|---|
| Fire Tablets | 2818 | 4149 chars |
| Alexa Devices | 1566 | 4669 chars |
| E-Readers & Accessories | 361 | 3966 chars |
| E-Readers Premium | 56 | 4256 chars |

> Streaming Devices excluded — only 4 reviews available.

### ✅ Conclusion
GPT-4o-mini successfully generated structured recommendation articles
for all 4 main product categories based on real customer review data.

Each article includes:
- 🏆 Top 3 products with key differences
- ⚠️ Main complaints per product
- ❌ Product to avoid with justification
- 💡 Final recommendation per user type

Articles are saved and ready for display in the web app.